# Tratamento: ZS_FUT e BRL_BRL
Tratamento de dados agrícolas com limpeza de valores nulos e criação de lags da coluna de fechamento.

In [1]:
import pandas as pd
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
print('✓ Bibliotecas carregadas.')

✓ Bibliotecas carregadas.


In [2]:
BASE_DIR = Path('.')
ARQUIVO_ZS = BASE_DIR / 'ZS_FUT.csv'
ARQUIVO_BRL = BASE_DIR / 'BRL_BRL.csv'
PASTA_SAIDA = BASE_DIR / 'indicador_agro_tratado'
PASTA_SAIDA.mkdir(exist_ok=True)
print('Caminhos configurados.')

Caminhos configurados.


In [3]:
def arredondar_moeda(valor, casas=4):
    if pd.isna(valor):
        return valor
    d = Decimal(str(valor))
    q = '0.' + '0' * casas
    return float(d.quantize(Decimal(q), rounding=ROUND_HALF_UP))

def preparar_dataset(df, colunas_preco=('Close', 'High', 'Low', 'Open'), lags=(1, 3, 6, 10)):
    # Garante ordenacao cronologica antes de criar lags.
    df = df.sort_index()

    for col in colunas_preco:
        df[col] = df[col].apply(arredondar_moeda)

    df.columns = df.columns.str.lower()
    df.index.name = 'data'

    for lag in lags:
        df[f'close_lag_{lag}d'] = df['close'].shift(lag)

    # Remove qualquer linha com celula vazia apos criacao dos lags.
    df = df.dropna(how='any')

    return df

print('✓ Funcoes de arredondamento e lags definidas.')

✓ Funcoes de arredondamento e lags definidas.


## Parte 1: ZS_FUT

In [4]:
df_zs = pd.read_csv(ARQUIVO_ZS, skiprows=2, names=['Data', 'Close', 'High', 'Low', 'Open', 'Volume'])
df_zs['Data'] = pd.to_datetime(df_zs['Data'], errors='coerce')
df_zs = df_zs.dropna(subset=['Data']).set_index('Data').sort_index()
colunas_preco = ['Close', 'High', 'Low', 'Open']
df_zs[colunas_preco] = df_zs[colunas_preco].replace(r'^\s*$', pd.NA, regex=True)
df_zs[colunas_preco] = df_zs[colunas_preco].apply(pd.to_numeric, errors='coerce')
df_zs = df_zs.dropna(subset=colunas_preco)
print(f'ZS_FUT lido: {df_zs.shape}')

ZS_FUT lido: (1760, 5)


/tmp/ipykernel_6274/2768482526.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_zs['Data'] = pd.to_datetime(df_zs['Data'], errors='coerce')


In [5]:
df_zs = preparar_dataset(df_zs)
print(f'ZS_FUT processado: {df_zs.shape}')
print('Colunas de lag:', [c for c in df_zs.columns if c.startswith('close_lag_')])

ZS_FUT processado: (1750, 9)
Colunas de lag: ['close_lag_1d', 'close_lag_3d', 'close_lag_6d', 'close_lag_10d']


In [6]:
df_zs.to_csv(PASTA_SAIDA / 'Soja_Chicago_USD_Bushel.csv')
print(f'✓ ZS_FUT salvo: {(PASTA_SAIDA / "Soja_Chicago_USD_Bushel.csv").stat().st_size} bytes')

✓ ZS_FUT salvo: 132552 bytes


## Parte 2: BRL_BRL

In [7]:
df_brl = pd.read_csv(ARQUIVO_BRL, skiprows=2, names=['Data', 'Close', 'High', 'Low', 'Open', 'Volume'])
df_brl['Data'] = pd.to_datetime(df_brl['Data'], errors='coerce')
df_brl = df_brl.dropna(subset=['Data']).set_index('Data').sort_index()
df_brl = df_brl[['Close', 'High', 'Low', 'Open']]
df_brl = df_brl.replace(r'^\s*$', pd.NA, regex=True)
df_brl = df_brl.apply(pd.to_numeric, errors='coerce')
df_brl = df_brl.dropna()
print(f'BRL_BRL lido: {df_brl.shape}')

BRL_BRL lido: (1825, 4)


/tmp/ipykernel_6274/1284197117.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_brl['Data'] = pd.to_datetime(df_brl['Data'], errors='coerce')


In [8]:
df_brl = preparar_dataset(df_brl)
print(f'BRL_BRL processado: {df_brl.shape}')
print('Colunas de lag:', [c for c in df_brl.columns if c.startswith('close_lag_')])

BRL_BRL processado: (1815, 8)
Colunas de lag: ['close_lag_1d', 'close_lag_3d', 'close_lag_6d', 'close_lag_10d']


In [9]:
df_brl.to_csv(PASTA_SAIDA / 'Cambio_USD_BRL.csv')
print(f'✓ BRL_BRL salvo: {(PASTA_SAIDA / "Cambio_USD_BRL.csv").stat().st_size} bytes')

✓ BRL_BRL salvo: 119956 bytes


## Resumo

In [10]:
print('='*60)
print(f'ZS_FUT: {len(df_zs)} linhas, colunas {list(df_zs.columns)}')
print(f'BRL_BRL: {len(df_brl)} linhas, colunas {list(df_brl.columns)}')
print(f'Salvos em: {PASTA_SAIDA}')
print('='*60)

ZS_FUT: 1750 linhas, colunas ['close', 'high', 'low', 'open', 'volume', 'close_lag_1d', 'close_lag_3d', 'close_lag_6d', 'close_lag_10d']
BRL_BRL: 1815 linhas, colunas ['close', 'high', 'low', 'open', 'close_lag_1d', 'close_lag_3d', 'close_lag_6d', 'close_lag_10d']
Salvos em: indicador_agro_tratado
